# ComfyUI on Kaggle — tts-img

本笔记本只负责“选择配置 + 调用库”，公共逻辑全部在 `kgcomfy` 包里。

- 库位置：`/kaggle/input/kg-notebook-lib/src/kgcomfy`（把仓库上传为 Kaggle Dataset）
- 默认配置：`src/kgcomfy/configs/tts-img.yaml`
- 需要改模型路径、更新开关、frp 子域名时，优先改下面的 `overrides`，或在仓库里改 YAML
- 原笔记本保留在 `legacy/tts-img.original.ipynb`，方便对照

## 首次使用前

1. 在 Kaggle 笔记本的 **Add-ons → Secrets** 中添加 `frp_token`（可选：`modelscope_token`）
2. 把本仓库上传为 Kaggle Dataset，名称建议 `kg-notebook-lib`
3. 按需在 Notebook 设置里挂载模型 Dataset / 输入数据


## 0. 加载 kgcomfy 库

In [ ]:
import os
import sys

# 支持两种分发方式：
# 1) 已通过 pip install 安装 kgcomfy
# 2) 把本仓库作为 Kaggle Dataset 挂载（默认路径 /kaggle/input/kg-notebook-lib）
try:
    import kgcomfy
except ImportError:
    CANDIDATES = [
        os.environ.get("KGCOMFY_HOME", ""),
        "/kaggle/input/kg-notebook-lib",
        "/kaggle/working/kg-notebook",
        os.getcwd(),
    ]

    for base in CANDIDATES:
        for root in (base, os.path.join(base, "src")):
            if root and os.path.isdir(os.path.join(root, "kgcomfy")):
                sys.path.insert(0, root)
                break
        else:
            continue
        break
    else:
        raise RuntimeError(
            "找不到 kgcomfy 库。请将本仓库上传为 Kaggle Dataset（名称 kg-notebook-lib），"
            "或设置环境变量 KGCOMFY_HOME 指向仓库根目录。"
        )

    import kgcomfy

print("kgcomfy", kgcomfy.__version__, "->", kgcomfy.__file__)


## 1. 配置

所有可能变化的项都在这里或 YAML 里，不需要改库代码。

In [ ]:
from kgcomfy import KaggleComfy, load_config

cfg = load_config(
    "tts-img.yaml",
    overrides={
        # 更新开关：需要最新 ComfyUI / 自定义节点时改成 True
        "updates": {
            "comfyui": False,
            "custom_nodes": False,
        },
        # frp：换子域名即可得到 <subdomain>.oracle2.120168.xyz
        "frp": {
            "subdomain": "tts",
            "server_addr": "oracle2.120168.xyz",
        },
        # 完全替换模型映射时使用（注意：会覆盖 YAML 中的 models 列表）
        # "models": {
        #     "/kaggle/input/.../model.safetensors": "models/clip/model.safetensors",
        # },
        # 调试时只打印命令、不真正执行
        # "dry_run": True,
    },
)

app = KaggleComfy(cfg)
print("ComfyUI dir :", cfg.runtime.comfy_dir)
print("Public URL  :", app.public_url())
print("File browser:", app.filebrowser_url())


## 2. 安装运行时

下载并解压 ComfyUI 与便携 Python 运行时，按配置补装缺失的自定义节点。
已存在的目录会自动跳过；`aria2c` 由 `runtime.apt_packages` 安装。


In [ ]:
app.install()


## 3. 可选更新

由 `updates` 开关控制，默认关闭：

- `updates.comfyui`：`git pull` ComfyUI，并按 `comfyui_requirements` 重装依赖
- `updates.custom_nodes`：对所有 `enabled` 的节点执行 `git pull`
- 没有 `.git` 的目录（例如整包解压出来的节点）会安全跳过


In [ ]:
app.update()


## 4. 模型映射

按 YAML 里的 `models` 列表建立软链。`dest` 是相对路径时以 `cfg.runtime.comfy_dir` 为基准。
`cleanup.enabled=false` 时 `app.cleanup()` 是空操作。


In [ ]:
app.cleanup()
app.link_models()

for line in app.list_models(20):
    print(line)


## 5. 环境自检

In [ ]:
app.diagnostics()


## 6. 启动 frp 与文件浏览器

两个服务都在守护线程里运行，不会阻塞当前 Cell。
文件浏览器地址为 `app.filebrowser_url()`。


In [ ]:
app.start_services()

print("ComfyUI     :", app.public_url())
print("File browser:", app.filebrowser_url())


## 7. 启动 ComfyUI（前台阻塞）

停止 ComfyUI 时中断本 Cell 即可。

In [ ]:
# 该 Cell 会一直运行
app.run_comfyui()


## 可选：一键完整流程

In [ ]:
# 等价于 install + update + cleanup + link_models + start_services + run_comfyui
# app.run()


## 可选：打包 / 上传到 ModelScope

密钥来自 Kaggle Secrets，不再写进 notebook。

In [ ]:
# pub = app.publisher()
# pub.create_tar("/kaggle/working/ComfyUI.tar", "/kaggle/ComfyUI")
# pub.modelscope_upload("/kaggle/working/ComfyUI.tar", "ComfyUI.tar")


## 配置速查

| 配置项 | 作用 |
| --- | --- |
| `updates.comfyui` | 是否更新 ComfyUI 本体 |
| `updates.custom_nodes` | 是否更新自定义节点 |
| `models` | Kaggle 输入 → ComfyUI models 的路径映射 |
| `frp.subdomain` | 公网子域名，最终地址为 `<subdomain>.<server_addr>` |
| `frp.server_addr` / `server_port` | frp 服务端 |
| `frp.proxies` | 留空时自动生成 ComfyUI 与 fslist 两个 proxy |
| `filebrowser.enabled` | 是否启动文件浏览器 |
| `secrets.frp_token` | Kaggle Secret 名称，也可用环境变量 `FRP_TOKEN` |
| `cleanup.enabled` | 是否在链接前清理旧模型路径/软链 |
| `dry_run` | 只打印命令，不执行 |

完整字段说明见仓库 `README.md`。
